# 2 · Table 1: main models

This page estimates Table 1 and the pooled friendship interaction from
`data/generated/main/analysis_dyads.csv`. [Data and
measures](01_data_and_measures.ipynb) explains how that input is prepared.

# The rank gradient in cooperation

## What we want to measure

Take one decision: child $i$ meets classmate $j$. Two things describe
the pair:

-   the **Social Rank GAP**, $\text{Rank}_i - \text{Rank}_j$, between
    $-1$ and $1$. It is positive when $i$ is the higher-ranked child of
    the two;
-   the **Cooperation GAP**, tokens $i$ received from $j$ minus tokens
    $i$ sent to $j$, between $-10$ and $10$. It is positive when $i$
    came out ahead.

If cooperation were blind to rank, the Cooperation GAP would be
unrelated to the rank gap. If lower-ranked children give more to
higher-ranked classmates than they get back, the Cooperation GAP rises
with the rank gap. The slope of that relation is the quantity of
interest.

Each pair appears twice in the data, once from each side, with both gaps
changing sign. This is what we want: every child is a sender in some
rows, and the model compares the same sender across her different
partners.

## Panel A: regression estimates

For a decision of sender $i$ toward partner $j$, in classroom $c$ and
round $t$:

$$
\text{Coop GAP}_{ijct} = \beta\, \text{RankGAP}_{ij} + \eta_i + \kappa_{c \times t} + u_{ijct}
$$

Each term has a job:

-   $\eta_i$, **sender fixed effects**. Some children are generous with
    everyone, others with no one. The sender fixed effect absorbs each
    child’s own level, so $\beta$ is estimated only from how the *same*
    child’s exchanges differ across her partners.
-   $\kappa_{c\times t}$, **classroom-by-round fixed effects**. A
    classroom may be more cooperative overall, and giving may drift
    during the session. These fixed effects compare only decisions made
    in the same classroom and round, the unit within which partners were
    drawn at random.
-   **Standard errors clustered by classroom.** Decisions from the same
    classroom share children, teachers and nomination networks, so they
    are not independent. Clustering by classroom (45 clusters) allows
    for any correlation within a classroom.

Because partners were assigned at random within each classroom and
round, the rank of the partner a child met is unrelated to her own
characteristics ([Figure 1D](03_figure1_partner_assignment.ipynb) checks
this). So $\beta$ can be read as how the same child’s exchanges change
when she happens to meet a higher- or lower-ranked classmate.

The model is estimated separately in five samples, one per column of
Table 1: all pairs, non-friends, mutual friends, and the two kinds of
one-sided friendships, seen from the side of the child who named the
other: a lower-ranked child naming a higher-ranked one (L→H) or the
reverse (H→L).

In [ ]:
here::i_am("notebooks/main/02_table1_main_models.qmd")
library(tidyverse)
library(lfe)      # felm(): linear models with many fixed effects
library(here)

The data are reused if `analysis_dyads` and `analysis_students` are
already in R. Otherwise, this step loads their CSV files from
`data/generated/main/`, or builds them from the raw inputs on first use.

In [ ]:
source(here("code", "00_load_main_data.R"), local = TRUE)
dyads <- analysis_dyads %>%
  mutate(class_round = paste(class_id, round, sep = "_"),
         across(c(sender_id, class_id, class_round), as.factor))

In [ ]:
main_data_status

dataset,rows,source
analysis_dyads,17804,Saved CSV files
analysis_students,859,Saved CSV files


In [ ]:
# Five samples, one per column of Table 1.
#
# One-sided friendships are seen from the side of the child who named the
# other ("Sender declares"), and split by who has the higher rank:
#   L -> H : the lower-ranked child names the higher-ranked one (rank_gap < 0)
#   H -> L : the higher-ranked child names the lower-ranked one (rank_gap > 0)
# Two of these 1,300 pairs have exactly the same rank (rank_gap = 0). As in
# the paper, they are counted in the H -> L column.

samples <- list(
  "All dyads"      = dyads,
  "Non-friends"    = dyads %>% filter(friendship == "Non-friends"),
  "Mutual friends" = dyads %>% filter(friendship == "Mutual"),
  "Asym. (L->H)"   = dyads %>% filter(friendship == "Sender declares", rank_gap < 0),
  "Asym. (H->L)"   = dyads %>% filter(friendship == "Sender declares", rank_gap >= 0)
)

sapply(samples, nrow)

In [ ]:
sapply(samples, nrow)
##      All dyads    Non-friends Mutual friends   Asym. (L->H)   Asym. (H->L) 
##          17804          13916           1288            894            406

The two one-sided samples are small (fewer than 1,000 decisions each,
and few decisions per classroom and round), so they use classroom fixed
effects instead of classroom-by-round fixed effects.

The function `felm()` from the package **lfe** estimates linear models
with many fixed effects. Its formula has four parts separated by `|`:
the regression, the fixed effects, instruments (none here, `0`) and the
variable that defines the clusters.

In [ ]:
# Linear regression with fixed effects, standard errors clustered by classroom.
#
#   Coop GAP_ij = beta * RankGAP_ij + sender FE + classroom-by-round FE
#
# felm formula:  outcome ~ regressor | fixed effects | (no IV) 0 | cluster
#
# The two asymmetric samples are small, so they use classroom FE instead of
# classroom-by-round FE.

with_class_round <- coop_gap ~ rank_gap | sender_id + class_round | 0 | class_id
with_class       <- coop_gap ~ rank_gap | sender_id + class_id    | 0 | class_id

models <- list(
  "All dyads"      = felm(with_class_round, data = samples[["All dyads"]]),
  "Non-friends"    = felm(with_class_round, data = samples[["Non-friends"]]),
  "Mutual friends" = felm(with_class_round, data = samples[["Mutual friends"]]),
  "Asym. (L->H)"   = felm(with_class,       data = samples[["Asym. (L->H)"]]),
  "Asym. (H->L)"   = felm(with_class,       data = samples[["Asym. (H->L)"]])
)

### Example: the non-friend model

For example, the summary below shows the model estimated among
non-friends. The `rank_gap` row reports its estimated Social Rank GAP
slope, classroom-clustered standard error, and p-value. The table below
brings this estimate together with the other four samples.

In [ ]:
summary(models[["Non-friends"]])
## 
## Call:
##    felm(formula = with_class_round, data = samples[["Non-friends"]]) 
## 
## Residuals:
##      Min       1Q   Median       3Q      Max 
## -15.4848  -2.6283  -0.2574   2.6179  17.3630 
## 
## Coefficients:
##          Estimate Cluster s.e. t value Pr(>|t|)    
## rank_gap   2.5615       0.2813   9.104 1.11e-11 ***
## ---
## Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
## 
## Residual standard error: 4.3 on 12031 degrees of freedom
## Multiple R-squared(full model): 0.2968   Adjusted R-squared: 0.1866 
## Multiple R-squared(proj model): 0.02085   Adjusted R-squared: -0.1325 
## F-statistic(full model, *iid*):2.695 on 1884 and 12031 DF, p-value: < 2.2e-16 
## F-statistic(proj model): 82.89 on 1 and 44 DF, p-value: 1.111e-11

### Panel A results

The tibble below collects the estimated coefficients,
classroom-clustered standard errors, p-values, sample sizes, and model
fit for the five samples.

In [ ]:
# Panel A: coefficient, classroom-clustered uncertainty, and model fit.
table1_panel_a <- map_dfr(names(models), function(name) {
  model <- models[[name]]
  results <- summary(model)

  tibble(
    sample = name,
    beta = coef(model)[["rank_gap"]],
    se = results$coefficients["rank_gap", "Cluster s.e."],
    p_value = results$coefficients["rank_gap", "Pr(>|t|)"],
    n = model$N,
    r2 = results$r.squared,
    r2_adj = results$adj.r.squared
  )
})

table1_panel_a

In [ ]:
table1_panel_a

sample,beta,se,p_value,n,r2,r2_adj
All dyads,2.583957,0.2737866,0.0000000,17804,0.2752627,0.1891341
Non-friends,2.561512,0.2813463,0.0000000,13916,0.2967687,0.1866458
Mutual friends,1.069258,1.3273024,0.4248132,1288,0.8385785,0.0766692
Asym. (L->H),3.066224,1.1053284,0.0080931,894,0.6717628,0.2560511
Asym. (H->L),3.520170,3.6673080,0.3423617,406,0.8383592,0.3518364


## Panel B: effects in more intuitive units

A slope of $\beta$ tokens per unit of rank gap is hard to picture,
because a full unit (from the bottom to the top of a classroom) is rare.
Panel B expresses the same estimate in three other ways:

-   **in tokens**, for a typical difference in rank (one standard
    deviation of the Social Rank GAP);
-   **in standard deviations of the Cooperation GAP**, which makes the
    effect comparable across samples;
-   **model-adjusted**, using only the variation in the rank gap that
    remains after removing the fixed effects. This is the variation the
    model actually uses, so this number is the most conservative of the
    three.

In [ ]:
# Panel B puts beta in more intuitive units:
#
#   effect in tokens     = beta * SD(Social Rank GAP)
#   effect in SD tokens  = beta * SD(Social Rank GAP) / SD(Cooperation GAP)
#   model-adjusted       = beta * SD(Social Rank GAP after removing the fixed
#                          effects) / SD(Cooperation GAP)
#
# The last one uses only the part of the rank gap that the fixed-effects
# model actually uses (its variation within sender and within class-round).

fixed_effects_used <- list(
  "All dyads"      = c("sender_id", "class_round"),
  "Non-friends"    = c("sender_id", "class_round"),
  "Mutual friends" = c("sender_id", "class_round"),
  "Asym. (L->H)"   = c("sender_id", "class_id"),
  "Asym. (H->L)"   = c("sender_id", "class_id")
)

rank_gap_within <- function(data, fixed_effects) {
  # demeanlist() removes the fixed effects from a variable
  demeaned <- demeanlist(list(data$rank_gap), as.list(data[, fixed_effects]))
  demeaned[[1]]
}

### Panel B results

The tibble below contains the three effect-size transformations for each
sample.

In [ ]:
# Panel B: the same coefficient expressed in three standardized units.
table1_panel_b <- map_dfr(names(models), function(name) {
  model <- models[[name]]
  data <- samples[[name]]
  beta <- coef(model)[["rank_gap"]]

  tibble(
    sample = name,
    effect_tokens = beta * sd(data$rank_gap),
    effect_sd_tokens = beta * sd(data$rank_gap) / sd(data$coop_gap),
    effect_sd_model_adjust = beta * sd(rank_gap_within(data, fixed_effects_used[[name]])) /
                             sd(data$coop_gap)
  )
})

table1_panel_b

In [ ]:
table1_panel_b

sample,effect_tokens,effect_sd_tokens,effect_sd_model_adjust
All dyads,0.9159038,0.1930971,0.1314885
Non-friends,0.9234871,0.1936743,0.1223596
Mutual friends,0.3150996,0.0739164,0.0171756
Asym. (L->H),0.6706397,0.1438920,0.0891089
Asym. (H->L),0.6129626,0.1320735,0.0513552


## Table 1: estimates and standardized effects

We combine both panels in Table 1.

In [ ]:
stars <- function(p) {
  case_when(p < 0.001 ~ "***", p < 0.01 ~ "**", p < 0.05 ~ "*", TRUE ~ "")
}

table1 <- left_join(table1_panel_a, table1_panel_b, by = "sample")

table1 %>% mutate(across(where(is.numeric), ~ round(.x, 3)))

In [ ]:
# The same numbers, laid out as Table 1 (one column per sample)
table1_display <- table1 %>%
  transmute(
    sample,
    `Social Rank GAP`           = paste0(sprintf("%.2f", beta), stars(p_value)),
    `(SE)`                      = sprintf("(%.2f)", se),
    Observations                = format(n, big.mark = ",", trim = TRUE),
    `R2`                        = sprintf("%.3f", r2),
    `R2 adj.`                   = sprintf("%.3f", r2_adj),
    `Effect in tokens`          = paste0(sprintf("%.2f", effect_tokens), stars(p_value)),
    `Effect in SD tokens`       = paste0(sprintf("%.3f", effect_sd_tokens), stars(p_value)),
    `Effect in SD (model-adj.)` = paste0(sprintf("%.3f", effect_sd_model_adjust), stars(p_value))
  ) %>%
  pivot_longer(-sample, names_to = "row") %>%
  pivot_wider(names_from = sample, values_from = value)

table1_display

``` r
# Use the computed display values for each cell; formatting does not alter estimates.
table_value <- function(label) {
  cells <- unlist(table1_display[table1_display$row == label, -1], use.names = FALSE)
  cells <- htmltools::htmlEscape(cells)
  gsub("(\\*+)$", "<sup>\\1</sup>", cells, perl = TRUE)
}

table_row <- function(label, values, class = "") {
  paste0('<tr class="', class, '"><th scope="row">', label, '</th>',
         paste0('<td>', values, '</td>', collapse = ""), '</tr>')
}

panel_row <- function(label) {
  paste0('<tr class="panel-heading"><th colspan="6">', label, '</th></tr>')
}

table1_html <- paste0(
  '<style>
  .hierarchy-table1 { margin: 1.25rem 0; font-size: 0.9rem; line-height: 1.45; }
  .hierarchy-table1 .table-scroll { overflow-x: auto; }
  .hierarchy-table1 table { width: 100%; min-width: 710px; border-collapse: collapse; border-top: 2px solid #383838; border-bottom: 2px solid #383838; }
  .hierarchy-table1 caption { caption-side: top; color: inherit; text-align: left; padding: 0 0 0.75rem; }
  .hierarchy-table1 th, .hierarchy-table1 td { padding: 0.35rem 0.5rem; border: 0; background: transparent; }
  .hierarchy-table1 thead th { text-align: center; vertical-align: bottom; font-weight: 600; border-bottom: 1px solid #383838; }
  .hierarchy-table1 tbody th, .hierarchy-table1 tbody td[scope="row"] { text-align: left; font-weight: 400; white-space: normal; }
  .hierarchy-table1 tbody td { text-align: center; white-space: nowrap; font-variant-numeric: tabular-nums; }
  .hierarchy-table1 .panel-heading th, .hierarchy-table1 .panel-heading td { text-align: left; font-style: italic; font-weight: 600; border-top: 1px solid #383838; padding-top: 0.65rem; padding-bottom: 0.5rem; }
  .hierarchy-table1 .standard-errors th, .hierarchy-table1 .standard-errors td { padding-top: 0; padding-bottom: 0.65rem; }
  .hierarchy-table1 .table-notes { margin-top: 0.75rem; }
  .hierarchy-table1 sup { font-size: 0.75em; }
  </style>',
  '<div class="hierarchy-table1"><div class="table-scroll">',
  '<table aria-label="Table 1. Cooperation GAP models by relationship type">',
  '<caption><strong>Table 1.</strong> Status differences and cooperative asymmetries at the dyadic level by relationship type</caption>',
  '<thead><tr><th scope="col"></th>',
  '<th scope="col">All dyads</th><th scope="col">Non-friends</th>',
  '<th scope="col">Mutual friends</th><th scope="col">Asym.<br>(L&rarr;H)</th>',
  '<th scope="col">Asym.<br>(H&rarr;L)</th></tr></thead><tbody>',
  panel_row('Panel A: Dep. variable = Cooperation GAP (tokens received &minus; sent)'),
  table_row('Social Rank GAP', table_value('Social Rank GAP')),
  table_row('', table_value('(SE)'), 'standard-errors'),
  table_row('Observations', table_value('Observations')),
  table_row('R<sup>2</sup>', table_value('R2')),
  table_row('R<sup>2</sup> adj.', table_value('R2 adj.')),
  table_row('Sender FE', rep('Yes', 5)),
  table_row('Class &times; round FE', c('Yes', 'Yes', 'Yes', 'No', 'No')),
  table_row('Class FE', c('No', 'No', 'No', 'Yes', 'Yes')),
  table_row('Round FE', rep('No', 5)),
  panel_row('Panel B: Standardized effect (per 1 SD Social Rank GAP)'),
  table_row('Effect in tokens', table_value('Effect in tokens')),
  table_row('Effect in SD tokens', table_value('Effect in SD tokens')),
  table_row('Effect in SD tokens<br>(model-adjusted)', table_value('Effect in SD (model-adj.)')),
  '</tbody></table></div>',
  '<p class="table-notes"><em>Notes:</em> Each column reports a separate linear regression of the Cooperation GAP on the Social Rank GAP. Standard errors in parentheses are clustered by classroom. L&rarr;H denotes a lower-ranked child nominating a higher-ranked child; H&rarr;L denotes the reverse. Panel B transforms the corresponding coefficient from Panel A using the within-sample standard deviations; the model-adjusted effect uses the rank-gap variation remaining after the fixed effects. <sup>*</sup> p &lt; 0.05, <sup>**</sup> p &lt; 0.01, <sup>***</sup> p &lt; 0.001.</p>',
  '</div>'
)
```

``` r
if (isTRUE(getOption("knitr.in.progress"))) {
  knitr::asis_output(table1_html)
} else if (requireNamespace("IRdisplay", quietly = TRUE)) {
  IRdisplay::display_html(table1_html)
} else {
  cat(table1_html)
}
```

**How to read it.** Among non-friends, a one-unit increase in the Social
Rank GAP raises the Cooperation GAP by 2.56 tokens (SE 0.28, P \<
0.001), in favor of the higher-ranked child. Over the full range, from
the lowest- to the highest-ranked child of a classroom (a gap of 2),
this is about 5.1 tokens out of 10. Among mutual friends the slope is
1.07 and not distinguishable from zero. The column with all pairs is
close to the non-friends column, because non-friends are 78% of the
pairs.

In the one-sided friendships, the L→H slope (3.07) looks like the
non-friend slope. The H→L estimate is imprecise (SE 3.67), as this
sample is the smallest.

The $R^2$ is high in the small samples because the many fixed effects
explain much of the variation when each child contributes only a few
decisions; the adjusted $R^2$ corrects for the number of fixed effects.

## Is the slope flatter among mutual friends?

Separate regressions describe each group but do not test whether two
slopes differ. For that we pool non-friends and mutual friends in a
single model and let the slope change with friendship:

$$
\text{Coop GAP}_{ijct} = \beta_1 \text{RankGAP}_{ij} + \beta_2\, \text{Mutual}_{ij}
 + \beta_3 \,(\text{RankGAP}_{ij} \times \text{Mutual}_{ij}) + \eta_i + \kappa_{c\times t} + u_{ijct}
$$

$\beta_1$ is the slope among non-friends, and $\beta_1 + \beta_3$ the
slope among mutual friends. The test of interest is $\beta_3 = 0$.

In [ ]:
# Separate regressions describe each group but do not test whether the slopes
# differ. The pooled model does: non-friends and mutual friends together, with
# a mutual-friend dummy and its interaction with the rank gap.
#
#   Coop GAP = b1 RankGAP + b2 Mutual + b3 (RankGAP x Mutual) + sender FE + class-round FE
#
# b3 < 0 means the rank gradient is flatter among mutual friends.

non_and_mutual <- dyads %>%
  filter(friendship %in% c("Non-friends", "Mutual")) %>%
  mutate(mutual = as.integer(friendship == "Mutual"))

pooled <- felm(coop_gap ~ rank_gap * mutual | sender_id + class_round | 0 | class_id,
               data = non_and_mutual)

pooled_table <- summary(pooled)$coefficients %>%
  as_tibble(rownames = "term")

pooled_table

In [ ]:
pooled_table

term,Estimate,Cluster s.e.,t value,Pr(>|t|)
rank_gap,2.592434,0.2669649,9.7107667,0.0000000
mutual,-0.030037,0.0909858,-0.3301289,0.7428693
rank_gap:mutual,-1.673607,0.5573200,-3.0029550,0.0043973


The interaction is $\beta_3$ = -1.674 (P = 0.004): among mutual friends
the rank gradient is 0.92 tokens instead of 2.59, a reduction of about
65%. The level difference $\beta_2$ is close to zero. This is expected:
the two decisions of a pair have Cooperation GAPs of opposite sign, so a
characteristic shared by both children of the pair, such as being mutual
friends, can hardly shift the level of the Cooperation GAP; what it can
change is the slope.

In [ ]:
write_csv(table1,         here("output", "tables", "table1.csv"))
write_csv(table1_display, here("output", "tables", "table1_display.csv"))
write_csv(pooled_table,   here("output", "tables", "table1_pooled_interaction.csv"))

# Session information

In [ ]:
sessionInfo()
## R version 4.3.2 (2023-10-31)
## Platform: aarch64-apple-darwin20 (64-bit)
## Running under: macOS 26.7.1
## 
## Matrix products: default
## BLAS:   /Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/lib/libRblas.0.dylib 
## LAPACK: /Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/lib/libRlapack.dylib;  LAPACK version 3.11.0
## 
## locale:
## [1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8
## 
## time zone: America/Santiago
## tzcode source: internal
## 
## attached base packages:
## [1] stats     graphics  grDevices utils     datasets  methods   base     
## 
## other attached packages:
##  [1] lfe_3.1.1       Matrix_1.6-1.1  lubridate_1.9.4 forcats_1.0.0  
##  [5] stringr_1.5.1   dplyr_1.1.4     purrr_1.0.4     readr_2.1.5    
##  [9] tidyr_1.3.1     tibble_3.2.1    ggplot2_3.5.2   tidyverse_2.0.0
## [13] here_1.0.1     
## 
## loaded via a namespace (and not attached):
##  [1] sandwich_3.1-1    generics_0.1.3    stringi_1.8.7     lattice_0.22-6   
##  [5] hms_1.1.3         digest_0.6.37     magrittr_2.0.3    evaluate_1.0.3   
##  [9] grid_4.3.2        timechange_0.3.0  fastmap_1.2.0     rprojroot_2.0.4  
## [13] jsonlite_2.0.0    Formula_1.2-5     scales_1.3.0      cli_3.6.4        
## [17] rlang_1.1.6       crayon_1.5.3      bit64_4.6.0-1     munsell_0.5.1    
## [21] withr_3.0.2       yaml_2.3.10       tools_4.3.2       parallel_4.3.2   
## [25] tzdb_0.5.0        colorspace_2.1-1  vctrs_0.6.5       R6_2.6.1         
## [29] zoo_1.8-13        lifecycle_1.0.4   bit_4.6.0         vroom_1.6.5      
## [33] pkgconfig_2.0.3   pillar_1.10.2     gtable_0.3.6      glue_1.8.0       
## [37] xfun_0.52         tidyselect_1.2.1  knitr_1.50        xtable_1.8-4     
## [41] htmltools_0.5.8.1 rmarkdown_2.29    compiler_4.3.2

[Continue to Figure 1D: partner
assignment](03_figure1_partner_assignment.ipynb).